###Faisal Alsuqayri

---

Assignment 2

In [1]:
%pip install -q langchain langchain_openai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 926.1 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install tavily-python

In [3]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

In [4]:
try:
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except ImportError:
    pass

In [5]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=userdata.get("TAVILY_API_KEY"))

In [6]:
# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [7]:
import requests
from langchain.tools import tool

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    try:
        response = requests.get(url, timeout=10.0)
        response.raise_for_status()
        return response.text[:5000]
    except Exception as e:
        return f"Error fetching {url}: {str(e)}"

In [8]:
from typing import Literal


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [12]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """You are an expert research assistant. To answer the user's question, you MUST follow these steps exactly:
1. Refine the user's query into a strong search engine query.
2. Use the 'internet_search' tool with your refined query to get the top 3 results.
3. Extract the URLs from those results and use the 'fetch_url' tool to read the text content of EACH of the 3 URLs.
4. Synthesize the information you read from the websites to provide a comprehensive, accurate answer to the user's original question.
5. Cite your sources by including the URLs at the end.
"""

tools_list = [internet_search, fetch_url]

my_research_agent = create_agent(
    model=model_nemotron3_nano,
    tools=tools_list,
    system_prompt=system_prompt
)

In [13]:
user_question = "What are the latest features of LangGraph?"

result = my_research_agent.invoke({
    "messages": [
        HumanMessage(user_question)
    ]
})

In [14]:
print(result["messages"][-1].content)

**Latest features of LangGraph (as of the most recent updates in 2025)**  

1. **Production‑ready core capabilities** – LangGraph 1.0 (released in October 2025) introduces durable execution, native streaming support, built‑in human‑oversight mechanisms, and persistent memory, making it suitable for production AI workflows.  

2. **Deferred node execution** – A new execution model lets you define nodes that run only after certain conditions or downstream nodes have completed, enabling more sophisticated, conditional flows.  

3. **Enhanced flow control & composability** – The deferred execution feature works alongside existing control‑flow constructs, allowing you to build complex, multi‑step pipelines with fine‑grained timing and dependency management.  

4. **Improved observability & debugging** – The release includes better tracing and monitoring hooks, helping engineers inspect intermediate states and reason about long‑running graph executions.  

5. **Memory & state management upgr